# XGBoost Wide Gridsearch — Honest CV (Stage 3 replacement)

Stage 3 rankings were anti-correlated with honest CV rankings. This is a clean re-run.

**CV protocol:** for each test year Y: ES on Y-1 → `best_round`, retrain on 2015..Y-1, test on Y.

**Informed by XGB_tuning2 (honest CV):**
- depth=8 & depth=5 led; depth=3 was worst → shift range up to [5,6,7,8,9]
- mcw=3 dominated top 20; mcw=1 never appeared → use [2,3,6]
- gamma>0 consistently better; 0.1 was sweet spot → include [0.0, 0.1, 0.5]

**Fixed:** `subsample=0.9`, `reg_alpha=0`

**Grid:** 5 × 3 × 3 × 3 × 3 × 3 = **1,215 combos** × 2 trains × 5 folds = 12,150 XGBoost fits

In [ ]:
import warnings, itertools
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('../..').resolve()
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold' / 'stage1' / 'hM7_Linj14_tau150_hT7'

LABEL_COL    = 'home_win'
CV_YEARS     = list(range(2020, 2025))
CLIP_EPS     = 1e-7
N_PLAYERS    = 7

# Fixed params — subsample=0.8 (no valid prior, use standard default)
FIXED = {
    'objective':   'binary:logistic',
    'eval_metric': 'logloss',
    'subsample':   0.8,
    'reg_alpha':   0.0,
    'seed':        42,
}
NUM_BOOST_ROUND = 3000
EARLY_STOPPING  = 150
N_WORKERS = 4
NTHREAD   = 4

# Full fresh grid — no reliable priors on any model param
MAX_DEPTH_VALUES = [3, 4, 5, 6, 7, 8]      # full range: Stage 1 favoured d=3, tuning2 favoured d=8
MIN_CW_VALUES    = [1, 2, 3, 6]             # include 1 (excluded based on now-invalid tuning2)
GAMMA_VALUES     = [0.0, 0.1, 0.5, 1.0]    # include 1.0 (excluded based on now-invalid tuning2)
COL_BT_VALUES    = [0.6, 0.8, 1.0]
REG_L_VALUES     = [0.5, 1, 2]
LR_VALUES        = [0.01, 0.02, 0.03]

grid = list(itertools.product(MAX_DEPTH_VALUES, MIN_CW_VALUES, GAMMA_VALUES,
                               COL_BT_VALUES, REG_L_VALUES, LR_VALUES))
print(f'{len(grid)} combos x {len(CV_YEARS)} folds x 2 trains = {len(grid)*len(CV_YEARS)*2:,} XGBoost fits')
print(f'Parallelism: {N_WORKERS} workers x nthread={NTHREAD}')

2592 combos x 5 folds x 2 trains = 25,920 XGBoost fits
Parallelism: 4 workers x nthread=4


## Data prep & fold cache

In [16]:
PLAYER_MODEL_FEATURES = [
    'm_ewma_pre', 'q_pre', 'days_since_first_report_pre', 'days_since_last_dnp_pre',
    'consec_dnps_pre', 'played_last_game_pre', 'minutes_last_game_pre',
    'days_since_last_played_pre', 'injury_present_flag_pre',
]
RECENT_FORM_FEATURES = ['net_rtg_ewma_pre', 'efg_ewma_pre', 'tov_pct_ewma_pre', 'orb_pct_ewma_pre', 'ftr_ewma_pre']
STYLE_FEATURES       = ['off_3pa_rate_pre', 'def_3pa_allowed_pre', 'off_2pa_rate_pre', 'def_2pa_allowed_pre', 'off_tov_pct_pre', 'def_forced_tov_pre']
SCHEDULE_FEATURES    = ['days_rest_pre', 'is_b2b_pre', 'games_last_4_days_pre', 'games_last_7_days_pre', 'travel_miles_pre', 'timezone_shift_hours_pre']

def build_feature_cols(n):
    cols = []
    for side in ('home', 'away'):
        for slot in range(1, n + 1):
            for f in PLAYER_MODEL_FEATURES:
                cols.append(f'{side}_p{slot}_{f}')
    for f in RECENT_FORM_FEATURES + STYLE_FEATURES + SCHEDULE_FEATURES:
        cols.append(f'home_{f}')
        cols.append(f'away_{f}')
    return cols

def load_year(yr):
    return pd.read_csv(GOLD_DIR / f'game_xgboost_input_{yr}_REGPST.csv')

def cold_start_mask(df):
    col = 'home_p1_m_ewma_pre'
    return df[col] != 0 if col in df.columns else pd.Series(True, index=df.index)

def make_dmatrix(df, feat_cols):
    avail = [c for c in feat_cols if c in df.columns]
    dm = xgb.DMatrix(df[avail].values.astype(float), label=df[LABEL_COL].values.astype(float),
                     feature_names=avail, missing=np.nan)
    dm.set_base_margin(df['base_margin'].values.astype(float))
    return dm

def log_loss(y, p):
    p = np.clip(p, CLIP_EPS, 1 - CLIP_EPS)
    return -float(np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

all_data  = {yr: load_year(yr) for yr in range(2015, 2025)}
feat_cols = build_feature_cols(N_PLAYERS)
print(f'Loaded data. {len(feat_cols)} feature cols.')

# Pre-build per-fold DMatrices (shared across all combo workers)
fold_cache = {}
for Y in CV_YEARS:
    def concat(yrs, cs_filter=False):
        parts = []
        for yr in yrs:
            df = all_data[yr]
            if cs_filter and yr == 2015:
                df = df[cold_start_mask(df)]
            parts.append(df)
        return pd.concat(parts, ignore_index=True)

    es_train_df  = concat(range(2015, Y - 1), cs_filter=True)
    full_train_df= concat(range(2015, Y),     cs_filter=True)

    fold_cache[Y] = {
        'dm_es_train':   make_dmatrix(es_train_df,     feat_cols),
        'dm_es_val':     make_dmatrix(all_data[Y - 1], feat_cols),
        'dm_full_train': make_dmatrix(full_train_df,   feat_cols),
        'dm_test':       make_dmatrix(all_data[Y],     feat_cols),
        'y_test':        all_data[Y][LABEL_COL].values.astype(float),
    }

print('Fold DMatrix cache built.')

Loaded data. 160 feature cols.
Fold DMatrix cache built.


## Gridsearch

In [17]:
def run_combo(depth, mcw, gamma, cbt, rl, lr):
    params = {**FIXED, 'max_depth': depth, 'min_child_weight': mcw,
              'gamma': gamma, 'colsample_bytree': cbt,
              'reg_lambda': rl, 'learning_rate': lr, 'nthread': NTHREAD}
    fold_lls, best_rounds = [], []
    for Y in CV_YEARS:
        fc = fold_cache[Y]
        m_es = xgb.train(params, fc['dm_es_train'], NUM_BOOST_ROUND,
                         evals=[(fc['dm_es_val'], 'val')],
                         early_stopping_rounds=EARLY_STOPPING, verbose_eval=False)
        best_round = m_es.best_iteration + 1
        m_final = xgb.train(params, fc['dm_full_train'], best_round, verbose_eval=False)
        p = m_final.predict(fc['dm_test'])
        fold_lls.append(log_loss(fc['y_test'], p))
        best_rounds.append(best_round)
    return depth, mcw, gamma, cbt, rl, lr, float(np.mean(fold_lls)), fold_lls, best_rounds

results = []
done    = 0
total   = len(grid)

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(run_combo, *combo): combo for combo in grid}
    for fut in as_completed(futures):
        depth, mcw, gamma, cbt, rl, lr, mean_ll, fold_lls, best_rounds = fut.result()
        results.append({'depth': depth, 'mcw': mcw, 'gamma': gamma,
                        'cbt': cbt, 'rl': rl, 'lr': lr,
                        'mean_ll': mean_ll, 'fold_lls': fold_lls,
                        'mean_best_round': int(np.mean(best_rounds)),
                        'min_best_round':  int(np.min(best_rounds))})
        done += 1
        if done % 100 == 0 or done == total:
            best = min(results, key=lambda r: r['mean_ll'])
            print(f'[{done:4d}/{total}]  best: d={best["depth"]} mcw={best["mcw"]} '
                  f'g={best["gamma"]} cbt={best["cbt"]} rl={best["rl"]} lr={best["lr"]} '
                  f'-> {best["mean_ll"]:.5f}')

print('\nDone.')

[ 100/2592]  best: d=3 mcw=1 g=1.0 cbt=0.8 rl=2 lr=0.03 -> 0.60104
[ 200/2592]  best: d=3 mcw=2 g=0.0 cbt=0.8 rl=0.5 lr=0.03 -> 0.60031
[ 300/2592]  best: d=3 mcw=2 g=0.0 cbt=0.8 rl=0.5 lr=0.03 -> 0.60031
[ 400/2592]  best: d=3 mcw=6 g=0.0 cbt=1.0 rl=1 lr=0.02 -> 0.60023
[ 500/2592]  best: d=4 mcw=1 g=0.0 cbt=0.8 rl=2 lr=0.02 -> 0.59938
[ 600/2592]  best: d=4 mcw=2 g=0.5 cbt=0.6 rl=0.5 lr=0.02 -> 0.59878
[ 700/2592]  best: d=4 mcw=2 g=0.5 cbt=0.6 rl=0.5 lr=0.02 -> 0.59878
[ 800/2592]  best: d=4 mcw=2 g=0.5 cbt=0.6 rl=0.5 lr=0.02 -> 0.59878
[ 900/2592]  best: d=4 mcw=2 g=0.5 cbt=0.6 rl=0.5 lr=0.02 -> 0.59878
[1000/2592]  best: d=4 mcw=2 g=0.5 cbt=0.6 rl=0.5 lr=0.02 -> 0.59878
[1100/2592]  best: d=4 mcw=2 g=0.5 cbt=0.6 rl=0.5 lr=0.02 -> 0.59878
[1200/2592]  best: d=4 mcw=2 g=0.5 cbt=0.6 rl=0.5 lr=0.02 -> 0.59878
[1300/2592]  best: d=4 mcw=2 g=0.5 cbt=0.6 rl=0.5 lr=0.02 -> 0.59878
[1400/2592]  best: d=4 mcw=2 g=0.5 cbt=0.6 rl=0.5 lr=0.02 -> 0.59878
[1500/2592]  best: d=6 mcw=2 g=0.1 cbt=0

## Results

In [19]:
res_df = (pd.DataFrame(results)
            .sort_values('mean_ll')
            .reset_index(drop=True))

for i, yr in enumerate(CV_YEARS):
    res_df[f'll_{yr}'] = res_df['fold_lls'].apply(lambda x: round(x[i], 5))
res_df = res_df.drop(columns='fold_lls')

cols = ['depth','mcw','gamma','cbt','rl','lr','mean_ll','mean_best_round','min_best_round'] + [f'll_{y}' for y in CV_YEARS]

print('=== Top 25 ===')
print(res_df[cols].head(25).to_string(index=False))

print('\n=== Best per depth ===')
print(res_df.groupby('depth').first().reset_index()[['depth','mcw','gamma','cbt','rl','lr','mean_ll','mean_best_round']].to_string(index=False))

print('\n=== Best per learning_rate ===')
print(res_df.groupby('lr').first().reset_index()[['lr','depth','mcw','gamma','cbt','rl','mean_ll']].to_string(index=False))

print('\n=== Best per gamma ===')
print(res_df.groupby('gamma').first().reset_index()[['gamma','depth','mcw','cbt','rl','lr','mean_ll']].to_string(index=False))

print('\n=== Best per colsample_bytree ===')
print(res_df.groupby('cbt').first().reset_index()[['cbt','depth','mcw','gamma','rl','lr','mean_ll']].to_string(index=False))

w = res_df.iloc[0]
print(f'\n=== WINNER ===')
print(f'  depth={w.depth}  mcw={w.mcw}  gamma={w.gamma}  cbt={w.cbt}  rl={w.rl}  lr={w.lr}')
print(f'  mean_ll={w.mean_ll:.5f}  mean_best_round={w.mean_best_round}  min_best_round={w.min_best_round}')

=== Top 25 ===
 depth  mcw  gamma  cbt  rl   lr  mean_ll  mean_best_round  min_best_round  ll_2020  ll_2021  ll_2022  ll_2023  ll_2024
     6    3    1.0  0.6 1.0 0.03 0.597487               42               2  0.56977  0.60072  0.62517  0.59820  0.59357
     6    3    0.1  0.6 1.0 0.02 0.597870               64              39  0.58803  0.59788  0.61340  0.59725  0.59281
     6    3    0.1  0.6 0.5 0.01 0.598125              104              39  0.59217  0.60182  0.61334  0.59761  0.58569
     8    3    0.1  0.6 1.0 0.02 0.598146               57              29  0.58274  0.59666  0.62260  0.59689  0.59185
     7    3    0.5  0.6 1.0 0.02 0.598195               58               1  0.58729  0.59792  0.61705  0.59895  0.58977
     6    3    1.0  0.6 1.0 0.02 0.598229               62              29  0.58903  0.59997  0.61454  0.59666  0.59093
     8    3    0.0  0.6 2.0 0.02 0.598390               58              21  0.58552  0.60416  0.61238  0.60053  0.58936
     6    3    0.0  0.6 1

We go with second place due to instability in 1st place:
max_depth=6
min_child_weight=3
gamma=0.1
colsample_bytree=0.6
reg_lambda=1
learning_rate=0.02

## Experiment: very aggressive hyperparams

Deep trees, minimal regularization, high learning rate — territory completely outside the main grid.

In [20]:
# Very aggressive grid
AGG_DEPTH_VALUES = [8, 10, 12, 15]
AGG_MCW_VALUES   = [1]
AGG_GAMMA_VALUES = [0.0]
AGG_CBT_VALUES   = [0.8, 1.0]
AGG_RL_VALUES    = [0.01, 0.1, 0.5]
AGG_LR_VALUES    = [0.05, 0.1]

FIXED_AGG = {**FIXED, 'nthread': NTHREAD}

agg_grid = list(itertools.product(AGG_DEPTH_VALUES, AGG_MCW_VALUES, AGG_GAMMA_VALUES,
                                   AGG_CBT_VALUES, AGG_RL_VALUES, AGG_LR_VALUES))
print(f'{len(agg_grid)} combos x {len(CV_YEARS)} folds x 2 trains = {len(agg_grid)*len(CV_YEARS)*2:,} XGBoost fits')

48 combos x 5 folds x 2 trains = 480 XGBoost fits


In [23]:
def run_agg(depth, mcw, gamma, cbt, rl, lr):
    params = {**FIXED_AGG, 'max_depth': depth, 'min_child_weight': mcw,
              'gamma': gamma, 'colsample_bytree': cbt,
              'reg_lambda': rl, 'learning_rate': lr}
    fold_lls, best_rounds = [], []
    for Y in CV_YEARS:
        fc = fold_cache[Y]
        m_es = xgb.train(params, fc['dm_es_train'], NUM_BOOST_ROUND,
                         evals=[(fc['dm_es_val'], 'val')],
                         early_stopping_rounds=EARLY_STOPPING, verbose_eval=False)
        best_round = m_es.best_iteration + 1
        m_final = xgb.train(params, fc['dm_full_train'], best_round, verbose_eval=False)
        p = m_final.predict(fc['dm_test'])
        fold_lls.append(log_loss(fc['y_test'], p))
        best_rounds.append(best_round)
    return depth, mcw, gamma, cbt, rl, lr, float(np.mean(fold_lls)), fold_lls, best_rounds

agg_results = []
done = 0
total_agg = len(agg_grid)

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(run_agg, *combo): combo for combo in agg_grid}
    for fut in as_completed(futures):
        depth, mcw, gamma, cbt, rl, lr, mean_ll, fold_lls, best_rounds = fut.result()
        agg_results.append({'depth': depth, 'mcw': mcw, 'gamma': gamma,
                            'cbt': cbt, 'rl': rl, 'lr': lr,
                            'mean_ll': mean_ll, 'fold_lls': fold_lls,
                            'mean_best_round': int(np.mean(best_rounds)),
                            'min_best_round':  int(np.min(best_rounds))})
        done += 1
        if done % 12 == 0 or done == total_agg:
            best = min(agg_results, key=lambda r: r['mean_ll'])
            d=best['depth']; rl=best['rl']; lr=best['lr']; mll=best['mean_ll']; mbr=best['mean_best_round']
            print(f'[{done:2d}/{total_agg}]  best: d={d} rl={rl} lr={lr} -> {mll:.5f}  rounds={mbr}')

print('Done.')

[12/48]  best: d=8 rl=0.1 lr=0.05 -> 0.60313  rounds=1
[24/48]  best: d=10 rl=0.1 lr=0.05 -> 0.60119  rounds=7
[36/48]  best: d=10 rl=0.1 lr=0.05 -> 0.60119  rounds=7
[48/48]  best: d=15 rl=0.1 lr=0.1 -> 0.59985  rounds=2
Done.


In [25]:
MAIN_WINNER_LL = 0.597870   # rank-2 from main grid

agg_df = (pd.DataFrame(agg_results)
          .sort_values('mean_ll')
          .reset_index(drop=True))
agg_df.index += 1

ll_cols = [f'll_{y}' for y in CV_YEARS]
for i, r in enumerate(agg_df['fold_lls']):
    for j, y in enumerate(CV_YEARS):
        agg_df.loc[i+1, f'll_{y}'] = round(r[j], 5)

display_cols = ['depth','mcw','gamma','cbt','rl','lr','mean_ll','mean_best_round','min_best_round'] + ll_cols
pd.set_option('display.float_format', '{:.5f}'.format)
pd.set_option('display.width', 160)

print(f'=== All {len(agg_df)} aggressive configs (sorted by mean_ll) ===')
print(agg_df[display_cols].to_string())

best_agg = agg_df.iloc[0]
print(f'=== Best aggressive config ===')
print(f'  depth={best_agg.depth}  mcw={best_agg.mcw}  gamma={best_agg.gamma}  cbt={best_agg.cbt}  rl={best_agg.rl}  lr={best_agg.lr}')
print(f'  mean_ll={best_agg.mean_ll:.5f}  mean_best_round={best_agg.mean_best_round}  min_best_round={best_agg.min_best_round}')
print(f'  vs main winner: {best_agg.mean_ll - MAIN_WINNER_LL:+.5f}')

=== All 48 aggressive configs (sorted by mean_ll) ===
    depth  mcw   gamma     cbt      rl      lr  mean_ll  mean_best_round  min_best_round  ll_2020  ll_2021  ll_2022  ll_2023  ll_2024
1      15    1 0.00000 0.80000 0.10000 0.10000  0.59985                2               1  0.60218  0.59741  0.61315  0.59812  0.58841
2      15    1 0.00000 0.80000 0.10000 0.05000  0.60117                6               1  0.59704  0.60249  0.62225  0.59798  0.58609
3      10    1 0.00000 0.80000 0.10000 0.05000  0.60119                7               1  0.59568  0.60843  0.61577  0.59650  0.58958
4      10    1 0.00000 0.80000 0.10000 0.10000  0.60124                1               1  0.59843  0.60165  0.62021  0.59479  0.59114
5      12    1 0.00000 0.80000 0.50000 0.05000  0.60135                3               1  0.59789  0.61129  0.61028  0.59892  0.58838
6      10    1 0.00000 1.00000 0.10000 0.05000  0.60156                5               1  0.59614  0.60963  0.61362  0.59888  0.58952
7      1